# Gauss Eliminate Simulation

จำลองการทำงานของฟังก์ชัน `gauss_eliminate` จาก C++  
(`src/algebra/matrix/matrix_solver.cpp`)

## อัลกอริธึม: Gaussian Elimination with Partial Pivoting

ขั้นตอน:
1. สำหรับแต่ละคอลัมน์ pivot `col = 0..n-1`:
   - **หา pivot row**: แถวที่มี `|mat[row][col]|` มากที่สุดในช่วง `row ≥ col`
   - **swap แถว** pivot กับแถว `col`
   - **eliminate**: สำหรับทุกแถว `r > col`:
     - คำนวณ factor = `mat[r][col] / mat[col][col]`
     - ลบ `factor × แถว col` ออกจากแถว `r`
2. คืนค่า pivot เล็กที่สุด (ใช้ detect singular matrix)

**Complexity**: O(n² × m) ต่อ column → O(n³) รวม (สำหรับ n×n matrix)

**Loop structure**:
```
for col in 0..n-1:              ← outer loop
    for row in col..m-1:        ← pivot search
    swap(row[pivot], row[col])
    for r in col+1..m-1:        ← elimination
        factor = mat[r][col] / mat[col][col]
        for c in col..n:        ← innermost: update each element
            mat[r][c] -= factor * mat[col][c]
```

In [1]:
from __future__ import annotations
import copy

kPivotTol = 1e-12  # เกณฑ์ pivot ≈ 0 (singular)


def fmt_mat(mat: list[list[float]], n: int, label: str = "") -> None:
    """แสดงเมทริกซ์ augmented [A | b] แบบสวยงาม"""
    m = len(mat)
    if label:
        print(f"  [{label}]")
    for i in range(m):
        a_part = "  ".join(f"{v:>9.4g}" for v in mat[i][:n])
        b_part = f"{mat[i][n]:>9.4g}" if n < len(mat[i]) else ""
        sep = "  |  " if b_part else ""
        print(f"    row {i}: [ {a_part}{sep}{b_part} ]")
    print()

In [2]:
def gauss_eliminate(
    mat: list[list[float]],
    n: int,
    verbose: bool = True,
) -> float:
    """
    Gaussian elimination with partial pivoting (in-place)

    Parameters
    ----------
    mat : list[list[float]]
        augmented matrix [A | b], ขนาด m × (n+1)
        จะถูกแก้ไข in-place
    n : int
        จำนวนตัวแปร (คอลัมน์ A, ไม่นับคอลัมน์ b)
    verbose : bool
        แสดง step-by-step detail

    Returns
    -------
    float
        smallest absolute pivot พบระหว่างกระบวนการ
        ถ้าใกล้ 0 → matrix อาจ singular
    """
    m = len(mat)
    smallest_pivot = float("inf")

    if verbose:
        print("=" * 60)
        print(f"Gauss Eliminate: m={m} rows, n={n} variables")
        print("=" * 60)
        fmt_mat(mat, n, "Initial matrix")

    # ─── Outer loop: ทำ pivot ทุก column ───────────────────────
    for col in range(n):
        if verbose:
            print(f"{'─'*50}")
            print(f">>> PIVOT COLUMN col={col} (variable column)")

        # ── Step 1: หา pivot row (row ที่มี |value| มากสุด) ───────
        pivot_row = col
        max_val = abs(mat[col][col]) if col < m else 0.0

        if verbose:
            print(f"  Step 1: หา pivot row จาก row {col} ถึง {m-1}")
            print(f"    เริ่มต้น: pivot_row={col}, |mat[{col}][{col}]|={max_val:.4g}")

        for row in range(col + 1, m):
            val = abs(mat[row][col])
            if verbose:
                arrow = " ← NEW MAX" if val > max_val else ""
                print(f"    row {row}: |mat[{row}][{col}]| = {val:.4g}{arrow}")
            if val > max_val:
                max_val = val
                pivot_row = row

        if verbose:
            print(f"  → pivot_row = {pivot_row}, pivot_value = {max_val:.4g}")

        # ── Step 2: swap แถว ────────────────────────────────────
        if pivot_row != col:
            mat[col], mat[pivot_row] = mat[pivot_row], mat[col]
            if verbose:
                print(f"  Step 2: SWAP row {col} ↔ row {pivot_row}")
                fmt_mat(mat, n, "after swap")
        else:
            if verbose:
                print(f"  Step 2: ไม่ต้อง swap (pivot อยู่ที่ row {col} แล้ว)")

        # ── Track smallest pivot ─────────────────────────────────
        pivot_val = abs(mat[col][col])
        smallest_pivot = min(smallest_pivot, pivot_val)
        if verbose:
            print(f"  pivot_val = {pivot_val:.4g}  (smallest so far = {smallest_pivot:.4g})")
            if pivot_val < kPivotTol:
                print(f"  ⚠ pivot ≈ 0 → matrix อาจ singular!")

        # ── Step 3: Elimination ─────────────────────────────────
        if verbose:
            print(f"  Step 3: Eliminate แถวที่ {col+1} ถึง {m-1}")

        for r in range(col + 1, m):
            if abs(mat[col][col]) < kPivotTol:
                if verbose:
                    print(f"    pivot ≈ 0 → ข้าม elimination")
                break

            factor = mat[r][col] / mat[col][col]

            if verbose:
                print(f"    Eliminate row {r}:")
                print(f"      factor = mat[{r}][{col}] / mat[{col}][{col}]")
                print(f"             = {mat[r][col]:.4g} / {mat[col][col]:.4g} = {factor:.4g}")

            # innermost loop: อัปเดตทุก element ในแถว r
            old_row = mat[r][:]
            for c in range(col, n + 1):
                delta = factor * mat[col][c]
                mat[r][c] -= delta
                if verbose:
                    print(
                        f"        mat[{r}][{c}] = {old_row[c]:>9.4g}"
                        f" - {factor:.4g}×{mat[col][c]+delta:>9.4g}"
                        f" = {mat[r][c]:>9.4g}"
                    )

            if verbose:
                row_str = "  ".join(f"{v:>9.4g}" for v in mat[r][:n])
                print(f"      → row {r} now: [ {row_str}  |  {mat[r][n]:>9.4g} ]")

        if verbose:
            fmt_mat(mat, n, f"after col={col} elimination")

    if verbose:
        print("=" * 60)
        print("FINAL Row Echelon Form:")
        fmt_mat(mat, n)
        print(f"smallest_pivot = {smallest_pivot:.4g}")
        if smallest_pivot < kPivotTol:
            print("⚠ Matrix may be singular (smallest_pivot < kPivotTol)")
        else:
            print("✓ Matrix appears non-singular")

    return smallest_pivot

## Test Case 1: ระบบ 2×2 ปกติ

```
2x + 3y = 7
 x -  y = 1
```

คาดหวัง: row echelon form แล้วแก้ได้ x=2, y=1

In [3]:
mat1 = [
    [2.0,  3.0,  7.0],
    [1.0, -1.0,  1.0],
]
mat1_orig = copy.deepcopy(mat1)
sp1 = gauss_eliminate(mat1, n=2)

Gauss Eliminate: m=2 rows, n=2 variables
  [Initial matrix]
    row 0: [         2          3  |          7 ]
    row 1: [         1         -1  |          1 ]

──────────────────────────────────────────────────
>>> PIVOT COLUMN col=0 (variable column)
  Step 1: หา pivot row จาก row 0 ถึง 1
    เริ่มต้น: pivot_row=0, |mat[0][0]|=2
    row 1: |mat[1][0]| = 1
  → pivot_row = 0, pivot_value = 2
  Step 2: ไม่ต้อง swap (pivot อยู่ที่ row 0 แล้ว)
  pivot_val = 2  (smallest so far = 2)
  Step 3: Eliminate แถวที่ 1 ถึง 1
    Eliminate row 1:
      factor = mat[1][0] / mat[0][0]
             = 1 / 2 = 0.5
        mat[1][0] =         1 - 0.5×        3 =         0
        mat[1][1] =        -1 - 0.5×      4.5 =      -2.5
        mat[1][2] =         1 - 0.5×     10.5 =      -2.5
      → row 1 now: [         0       -2.5  |       -2.5 ]
  [after col=0 elimination]
    row 0: [         2          3  |          7 ]
    row 1: [         0       -2.5  |       -2.5 ]

───────────────────────────────────

## Test Case 2: ระบบ 3×3 — ต้องมี swap หลายครั้ง

```
 0x + 2y +  z = 5
 3x -  y + 2z = 8
-x +  y -  z = -3
```

col=0: pivot ที่ row 0 คือ 0 → ต้องหา max pivot (row 1 มี 3)

In [ ]:
mat2 = [
    [ 0.0,  2.0,  1.0,  5.0],
    [ 3.0, -1.0,  2.0,  8.0],
    [-1.0,  1.0, -1.0, -3.0],
]
sp2 = gauss_eliminate(mat2, n=3)

## Test Case 3: Matrix ที่ singular (rank-deficient)

```
x +  y = 2
2x + 2y = 4   ← แถวซ้ำ (linearly dependent)
```

คาดหวัง: แถว 2 กลายเป็น 0 ทั้งแถว → smallest_pivot ≈ 0

In [ ]:
mat3 = [
    [1.0, 1.0, 2.0],
    [2.0, 2.0, 4.0],
]
sp3 = gauss_eliminate(mat3, n=2)
assert sp3 < kPivotTol, f"ควรได้ smallest_pivot ≈ 0, แต่ได้ {sp3}"
print(f"✓ singular detected: smallest_pivot = {sp3}")

## Test Case 4: ระบบ overdetermined (มากสมการ กว่าตัวแปร)

3 สมการ, 2 ตัวแปร:
```
 x +  y = 3
2x -  y = 3
 x + 2y = 4
```

gauss_eliminate ทำงานได้ปกติ ให้ row echelon form สำหรับ n=2 columns

In [ ]:
mat4 = [
    [1.0,  1.0,  3.0],
    [2.0, -1.0,  3.0],
    [1.0,  2.0,  4.0],
]
sp4 = gauss_eliminate(mat4, n=2)

## Test Case 5: Numerical Stability — เปรียบเทียบ with/without partial pivoting

ตัวอย่างคลาสสิกที่ partial pivoting ช่วย:
```
ε·x + y = 1
 x  + y = 2
```
ถ้า ε เล็กมาก และไม่ swap → factor = 1/ε ใหญ่มาก → catastrophic cancellation

In [ ]:
import math

eps = 1e-15

# With partial pivoting (row with |1| > |ε| จะถูก swap ขึ้นมาก่อน)
mat5_pp = [
    [eps, 1.0, 1.0],
    [1.0, 1.0, 2.0],
]
print("=== With Partial Pivoting ===")
gauss_eliminate(mat5_pp, n=2)

# ผลลัพธ์ row 2: [0, mat[1][1] - factor*1, ...]
# factor = 1/ε → ยกเลิกได้ถ้าไม่ swap
# แต่ partial pivoting swap ขึ้นก่อน → factor เล็ก → stable
print(f"solution x = {mat5_pp[1][2] / mat5_pp[1][1]:.6f}  (expected ≈ 1.0)")
print(f"solution y check...")

## สรุป Loop Structure

```
gauss_eliminate(mat, n):
│
├── for col in range(n):                          ← O(n)
│   ├── for row in range(col+1, m):               ← O(m) หา max pivot
│   │       max_val = max(max_val, |mat[row][col]|)
│   ├── swap(mat[col], mat[pivot_row])             ← O(n) element swap
│   ├── smallest_pivot = min(smallest_pivot, |mat[col][col]|)
│   └── for r in range(col+1, m):                 ← O(m) eliminate
│           factor = mat[r][col] / mat[col][col]
│           for c in range(col, n+1):             ← O(n) update row
│               mat[r][c] -= factor * mat[col][c]
│
└── return smallest_pivot
```

| Test | สถานการณ์ | n | m | pivot swap? | ผลลัพธ์ |
|------|-----------|---|---|-------------|--------|
| 1 | ปกติ 2×2 | 2 | 2 | ไม่ | row echelon |
| 2 | 3×3, row=0 pivot=0 | 3 | 3 | ใช่ (col=0) | row echelon |
| 3 | singular 2×2 | 2 | 2 | ไม่ | smallest_pivot≈0 |
| 4 | overdetermined 3×2 | 2 | 3 | ใช่ | partial echelon |
| 5 | numerical stability | 2 | 2 | ใช่ (ε→1) | stable solution |